In [ ]:
import yfinance as yf
import akshare as ak

t = yf.Ticker("2208.HK")
print("price ccy:", t.fast_info.get("currency"), "| financial ccy:", t.info.get("financialCurrency"))
for k in ["income_stmt", "balance_sheet", "cashflow"]:
    df = getattr(t, k)
    print(k, df.shape, [str(c)[:10] for c in df.columns])

In [ ]:
need = {
    "income_stmt": ["EBIT", "EBITDA", "Interest Expense", "Net Income"],
    "balance_sheet": ["Total Debt", "Cash And Cash Equivalents", "Total Assets"],
}
for k, items in need.items():
    df = getattr(t, k)
    for i in items:
        ok = i in df.index and df.loc[i].notna().all()
        print(k, i, "OK" if ok else "MISSING/NaN")

In [ ]:
for name, sym in [("IS", "利润表"), ("BS", "资产负债表"), ("CF", "现金流量表")]:
    try:
        df = ak.stock_financial_report_sina(stock="sz002202", symbol=sym)
        print(name, df.shape)
        df.to_csv(f"../data/raw/goldwind_{name}_sina.csv", index=False, encoding="utf-8-sig")
    except Exception as e:
        print(name, "ERR", type(e).__name__, str(e)[:100])

In [ ]:
try:
    df = ak.stock_financial_hk_report_em(stock="02208", symbol="利润表", indicator="年度")
    print("HK-IS", df.shape)
except Exception as e:
    print("HK-IS ERR", type(e).__name__, str(e)[:100])

In [ ]:
bs = t.balance_sheet
want = ["Total Debt", "Long Term Debt", "Current Debt", "Cash And Cash Equivalents", "Total Assets"]
print("tidak ada di index:", [i for i in want if i not in bs.index])
print(bs.loc[[i for i in want if i in bs.index]].to_string())

In [ ]:
hk = ak.stock_financial_hk_report_em(stock="02208", symbol="利润表", indicator="年度")
print(hk.columns.tolist())
print(hk.head(12).to_string())

In [ ]:
import pandas as pd
IS = pd.read_csv("../data/raw/goldwind_IS_sina.csv")
print(IS.shape)
print(IS.columns.tolist()[:10])
print(IS.iloc[:12, :5].to_string())
print("kolom berisi 利息:", [c for c in IS.columns if "利息" in str(c)])
mask = IS.astype(str).apply(lambda c: c.str.contains("利息")).any(axis=1)
print(IS[mask].iloc[:, :5].to_string())

In [ ]:
want = ["Total Debt", "Long Term Debt", "Current Debt", "Cash And Cash Equivalents", "Total Assets"]
bs_yi = t.balance_sheet.loc[want, :].dropna(axis=1, how="all") / 1e8   # satuan: 亿元
print(bs_yi.to_string(float_format="{:,.2f}".format))

In [ ]:
IS = pd.read_csv("../data/raw/goldwind_IS_sina.csv")
want = ["营业总收入", "营业利润", "利润总额", "净利润", "财务费用", "利息费用", "利息支出", "利息收入"]
print("tidak ada:", [c for c in want if c not in IS.columns])
cols = [c for c in want if c in IS.columns]
annual = IS[IS["报告日"].astype(str).str.endswith("1231")].set_index("报告日")[cols]
annual = annual.apply(pd.to_numeric, errors="coerce") / 1e8
print(annual.head(10).to_string(float_format="{:,.2f}".format))

In [ ]:
yf_inc = t.income_stmt.loc[["EBIT", "EBITDA", "Interest Expense", "Net Income"]].dropna(axis=1, how="all") / 1e8
yf_inc.columns = [c.strftime("%Y") for c in yf_inc.columns]
print(yf_inc.to_string(float_format="{:,.2f}".format))

In [ ]:
BS = pd.read_csv("../data/raw/goldwind_BS_sina.csv")
CF = pd.read_csv("../data/raw/goldwind_CF_sina.csv")

def annual_view(df, keywords, n=5):
    cols = [c for c in df.columns if any(k in str(c) for k in keywords)]
    out = df[df["报告日"].astype(str).str.endswith("1231")].set_index("报告日")[cols]
    return (out.apply(pd.to_numeric, errors="coerce") / 1e8).head(n)

print(annual_view(BS, ["货币资金", "短期借款", "一年内到期", "长期借款", "应付债券", "租赁负债", "资产总计", "负债合计"]).to_string(float_format="{:,.2f}".format))
print(annual_view(CF, ["折旧", "摊销", "折耗"]).to_string(float_format="{:,.2f}".format))

In [ ]:
cf_em = ak.stock_cash_flow_sheet_by_report_em(symbol="SZ002202")
print(cf_em.shape)
da_cols = [c for c in cf_em.columns if ("DEPR" in c or "AMORT" in c) and not c.endswith("_YOY")]
print(da_cols)
sel = cf_em[cf_em["REPORT_DATE"].astype(str).str.contains("12-31")].set_index("REPORT_DATE")[da_cols]
print((sel.apply(pd.to_numeric, errors="coerce") / 1e8).head(5).to_string(float_format="{:,.2f}".format))

In [ ]:
cols = ["货币资金", "短期借款", "一年内到期的非流动负债", "长期借款", "应付债券", "租赁负债", "负债合计"]
latest = BS[BS["报告日"].astype(str).isin(["20260630", "20260331"])].set_index("报告日")[cols]
print((latest.apply(pd.to_numeric, errors="coerce") / 1e8).to_string(float_format="{:,.2f}".format))

In [ ]:
BS = pd.read_csv("../data/raw/goldwind_BS_sina.csv")
want = [c for c in ["货币资金", "应付债券", "租赁负债", "长期应付款"] if c in BS.columns]
chk = BS[BS["报告日"].astype(str).str.endswith("1231")].set_index("报告日")[want]
print((chk / 1e8).head(11).to_string(float_format="{:,.2f}".format, na_rep="NaN"))

In [ ]:
bs_em = ak.stock_balance_sheet_by_report_em(symbol="SZ002202")
cash_cols = [c for c in bs_em.columns if "MONETARY" in c and not c.endswith("_YOY")]
print(cash_cols)
sel = bs_em[bs_em["REPORT_DATE"].astype(str).str.contains("12-31")].set_index("REPORT_DATE")[cash_cols]
print((sel.apply(pd.to_numeric, errors="coerce") / 1e8).head(11).to_string(float_format="{:,.2f}".format))

In [ ]:
IS = pd.read_csv("../data/raw/goldwind_IS_sina.csv")
want = [c for c in ["营业利润", "投资收益", "公允价值变动收益", "资产处置收益", "其他收益", "营业外收入", "营业外支出", "资产减值损失", "信用减值损失"] if c in IS.columns]
chk = IS[IS["报告日"].astype(str).str.endswith("1231")].set_index("报告日")[want]
print((chk / 1e8).head(6).to_string(float_format="{:,.2f}".format, na_rep="NaN"))